# AMZN Prediction V4 — Volatility + calibrated ranges
อัปโหลด `AMZN_End_to_End_Results.zip` (ผลจาก V1) แล้ว Run all

**สิ่งที่เปลี่ยนจาก V3**
1. **Rolling conformal** ใช้ residual นอกตัวอย่างที่ realized แล้วเท่านั้น (แทน quantile จากข้อมูล train ที่ซ้อนทับ)
2. **Earnings features** (อยู่ในหน้าต่างประกาศงบหรือไม่ / วันนับจากงบล่าสุด) เพิ่มเข้า HAR
3. **Stacking แบบ nested** ด้วย NNLS (weights ใช้ข้อมูลในอดีตเท่านั้น) แทนการเฉลี่ยตรง ๆ
4. **Diebold–Mariano (Newey–West) + non-overlapping evaluation** ยืนยันว่าชนะ naive จริง
5. เลือก `SHRINK` ด้วย CV, coverage แยกตามปีและระดับ VIX, interval score
6. **Forecast log** สำหรับตรวจกับราคาจริงในอนาคต (holdout ที่แท้จริงเดียวที่เหลือ)

ไม่ใช่คำแนะนำการลงทุน ผลเป็นหลักฐานเชิงทิศทาง ไม่ใช่ค่ารับประกัน

In [ ]:
%pip -q install arch

In [ ]:
import pandas as pd, numpy as np, zipfile, warnings, shutil
from pathlib import Path
from scipy import stats
from scipy.optimize import nnls
from sklearn.linear_model import LinearRegression
from arch import arch_model
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
HORIZONS     = [21, 63, 126]
FIRST_FOLD   = 2017            # first OOS fold year (warm-up for conformal + stacking)
CV_START     = pd.Timestamp('2019-01-01')
HOLDOUT_DAYS = 365
SHRINK_GRID  = [0.0, 0.5, 1.0]
CAL_WINDOW   = 1000            # rolling conformal window (realized samples)
MIN_CAL      = 250
QUANTILES    = [0.025, 0.10, 0.25, 0.50, 0.75, 0.90, 0.975]
OUT = Path(AMZN_OUT); OUT.mkdir(exist_ok=True, parents=True)

class _FilesZip:
    def __init__(self, files): self.files = files
    def open(self, name): return io.BytesIO(self.files[name].encode('utf-8'))
z = _FilesZip(AMZN_FILES)
def read(name):
    return pd.read_csv(z.open(name), parse_dates=['Date']).sort_values('Date').drop_duplicates('Date', keep='last').set_index('Date')
p = read('01_amzn_ohlcv_adjusted.csv'); m = read('03_market_competitor_prices.csv').reindex(p.index)
assert ((p[['Open','High','Low','Close']] > 0).all(axis=1) & (p.High >= p[['Open','Close','Low']].max(axis=1)) & (p.Low <= p[['Open','Close','High']].min(axis=1))).all(), 'Invalid OHLC'
idx, close = p.index, p.Close
r = np.log(close).diff(); LAST = idx[-1]; HOLDOUT_START = LAST - pd.Timedelta(days=HOLDOUT_DAYS)
print(f'{len(p)} rows {idx[0].date()} -> {LAST.date()}  last close {close.iloc[-1]:.2f}  holdout from {HOLDOUT_START.date()}')

## 1. Features (ข้อมูล ณ วันปิดตลาด + ปฏิทินงบ)

In [ ]:
pk = (np.log(p.High / p.Low) ** 2) / (4 * np.log(2))
F = pd.DataFrame(index=idx); F['d'] = np.log(np.sqrt(pk) + 1e-6)
for name, w in [('w', 5), ('m', 21), ('q', 63), ('y', 252)]: F[name] = np.log(np.sqrt(pk.rolling(w).mean()))
F['vix'] = np.log(m['^VIX'] / 100 / np.sqrt(252)); F['absr5'] = np.log(r.abs().rolling(5).mean() + 1e-4)
HAR, HARX, HARE = ['d','w','m','q','y'], ['d','w','m','q','y','vix','absr5'], ['d','w','m','q','y','vix','absr5','earn_in','dse']

# Earnings: Amazon reports after the close -> price reaction is the NEXT trading day.
e = pd.read_csv(z.open('12_amzn_earnings_surprise.csv'))
ed = pd.to_datetime(e['Earnings Date'], utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None).dt.normalize()
ed = sorted(set(ed[ed >= idx[0] - pd.Timedelta(days=120)]))
def react_pos(d):   # position of first trading day after report date; future dates extend past the last bar
    if d <= LAST: return int(np.searchsorted(idx.values, np.datetime64(d), side='right'))
    return len(idx) - 1 + int(np.busday_count(LAST.date(), (d + pd.Timedelta(days=1)).date()))
RPOS = np.array(sorted(set(react_pos(d) for d in ed)))
pos = np.arange(len(idx))
def earn_feats(h):
    nxt = np.searchsorted(RPOS, pos, side='right')                 # first reaction strictly after t
    earn_in = ((nxt < len(RPOS)) & (RPOS[np.minimum(nxt, len(RPOS)-1)] <= pos + h)).astype(float)
    prev = np.searchsorted(RPOS, pos, side='right') - 1            # last reaction <= t
    dse = np.where(prev >= 0, pos - RPOS[np.maximum(prev, 0)], 252)
    return earn_in, np.log1p(np.minimum(dse, 252))
FH, YV, RET, END = {}, {}, {}, {}
for h in HORIZONS:
    f = F.copy(); f['earn_in'], f['dse'] = earn_feats(h); FH[h] = f
    YV[h]  = np.log(np.sqrt((r ** 2).rolling(h).mean().shift(-h)))
    RET[h] = np.log(close.shift(-h) / close)
    END[h] = pd.Series(idx, index=idx).shift(-h)
EWMA = np.log(np.sqrt((r ** 2).ewm(alpha=0.06).mean()))
NAIVE = np.log(np.sqrt((r ** 2).rolling(21).mean()))
print('earnings reactions in data:', len(RPOS), '| next report:', ed[-1].date())

## 2. Out-of-sample volatility forecasts (expanding window, purge ตามวันสิ้นสุด target)

In [ ]:
BASE = ['har', 'har_vix', 'har_e', 'gjr']
def vol_predictions(h, fold_start, fold_stop, train_mask):
    te = (idx >= fold_start) & (idx < fold_stop); Fh = FH[h]; out = {}
    out['naive'] = NAIVE[te]; out['ewma'] = EWMA[te]
    for name, cols in [('har', HAR), ('har_vix', HARX), ('har_e', HARE)]:
        ok_tr = train_mask & Fh[cols].notna().all(axis=1).values
        lr = LinearRegression().fit(Fh[ok_tr][cols], YV[h][ok_tr])
        ok_te = te & Fh[cols].notna().all(axis=1).values
        out[name] = pd.Series(lr.predict(Fh[ok_te][cols]), index=idx[ok_te])
    last_train = int(np.flatnonzero(train_mask)[-1]); y = (r * 100).iloc[1:]
    res = arch_model(y.iloc[:last_train], vol='GARCH', p=1, o=1, q=1, dist='t', mean='Zero').fit(disp='off')
    fx = arch_model(y, vol='GARCH', p=1, o=1, q=1, dist='t', mean='Zero').fix(res.params)
    fc = fx.forecast(horizon=h, start=y.index[0], reindex=True).variance
    out['gjr'] = np.log(np.sqrt(fc.mean(axis=1)) / 100).reindex(idx)[te].dropna()
    return out

D = {}
for h in HORIZONS:
    frames = []
    for yr in range(FIRST_FOLD, LAST.year + 1):
        s, e_ = pd.Timestamp(f'{yr}-01-01'), min(pd.Timestamp(f'{yr+1}-01-01'), LAST + pd.Timedelta(days=1))
        tr = ((END[h] < s) & YV[h].notna() & FH[h][HARE].notna().all(axis=1)).values
        if tr.sum() < 300: continue
        d = pd.DataFrame(vol_predictions(h, s, e_, tr)); d['fold'] = yr; d['mu_raw'] = RET[h][tr].mean(); frames.append(d)
    d = pd.concat(frames).sort_index(); d['y'] = YV[h]; d['ret'] = RET[h]; d['end'] = END[h]
    # nested NNLS stacking: weights for year Y use only OOS predictions whose targets ended before Y
    d['stack'] = np.nan; W = {}
    for yr in sorted(d.fold.unique()):
        if yr < CV_START.year: continue
        trn = d[(d.fold < yr) & (d.end < pd.Timestamp(f'{yr}-01-01')) & d.y.notna()].dropna(subset=BASE)
        if len(trn) < 300: continue
        w, _ = nnls(trn[BASE].values, trn.y.values); W[yr] = w
        mk = (d.fold == yr).values; d.loc[mk, 'stack'] = d.loc[mk, BASE].values @ w
    D[h] = d; print(h, 'OOS rows', len(d), '| last-year stack weights', dict(zip(BASE, np.round(W[max(W)], 2))))

## 3. เลือกโมเดล (CV เท่านั้น) + Diebold–Mariano เทียบ naive

In [ ]:
MODELS = ['naive', 'ewma', 'har', 'har_vix', 'har_e', 'gjr', 'stack']
def dm_test(e1, e2, h):    # H1: model 2 has lower squared error than model 1; Newey-West lag h-1
    d = (e1 ** 2 - e2 ** 2).values; n = len(d); dm = d.mean(); s = ((d - dm) ** 2).mean()
    for k in range(1, h): s += 2 * (1 - k / h) * ((d[k:] - dm) * (d[:-k] - dm)).mean()
    t = dm / np.sqrt(s / n); return t, 1 - stats.norm.cdf(t)
rows = []
for h in HORIZONS:
    d = D[h]; cv = d[(d.index >= CV_START) & (d.end < HOLDOUT_START) & d.y.notna()].dropna(subset=MODELS)
    for k in MODELS:
        err = cv.y - cv[k]; no = cv.iloc[::h]; eno = no.y - no[k]
        t, pv = dm_test(cv.y - cv['naive'], err, h) if k != 'naive' else (np.nan, np.nan)
        rows.append(dict(horizon=h, model=k, rmse=np.sqrt((err ** 2).mean()), rmse_nonoverlap=np.sqrt((eno ** 2).mean()), corr=np.corrcoef(cv.y, cv[k])[0, 1], dm_t_vs_naive=t, dm_p_vs_naive=pv, n=len(cv), n_nonoverlap=len(no)))
cvs = pd.DataFrame(rows); cvs.to_csv(OUT / 'cv_vol_scores.csv', index=False)
BEST = {h: cvs[(cvs.horizon == h) & (cvs.model.isin(['har','har_vix','har_e','gjr','stack']))].sort_values('rmse').iloc[0].model for h in HORIZONS}
display(cvs.round(4)); print('Selected:', BEST)

## 4. Rolling conformal intervals + เลือก SHRINK ด้วย CV
z = (return − μ) / (σ̂·√h) เก็บเฉพาะที่ **target ปิดแล้วก่อนวันนั้น** แล้วใช้ quantile ของ z ล่าสุด `CAL_WINDOW` ตัว เทียบกับช่วงความกว้างคงที่ (ไม่ใช้ vol)

In [ ]:
def eval_intervals(h, model, shrink):
    d = D[h].dropna(subset=[model]).copy(); mu = shrink * d.mu_raw; sig = np.exp(d[model]) * np.sqrt(h)
    zr = ((d.ret - mu) / sig).values; rr = (d.ret - mu).values
    endv = d.end.fillna(pd.Timestamp('2100-01-01')).values; kk = np.searchsorted(endv, d.index.values, side='right')
    nq = len(QUANTILES); Z = np.full((len(d), nq), np.nan); C = np.full((len(d), nq), np.nan)
    for i, k in enumerate(kk):
        lo = max(0, k - CAL_WINDOW)
        if k - lo >= MIN_CAL: Z[i] = np.quantile(zr[lo:k], QUANTILES); C[i] = np.quantile(rr[lo:k], QUANTILES)
    o = pd.DataFrame(index=d.index); o['ret'] = d.ret; o['mu'] = mu; o['sig'] = sig; o['end'] = d.end; o['vix'] = m['^VIX'].reindex(d.index)
    for j, q in enumerate(QUANTILES):
        o[f'z{q}'] = Z[:, j]; o[f'q{q}'] = mu + Z[:, j] * sig; o[f'c{q}'] = mu + C[:, j]
    return o
def pin(y, q, a): return np.maximum(a * (y - q), (a - 1) * (y - q))
def metrics(o, pre):
    o = o.dropna(subset=['ret', f'{pre}0.1']); y = o.ret
    l, u = o[f'{pre}0.1'], o[f'{pre}0.9']; l95, u95 = o[f'{pre}0.025'], o[f'{pre}0.975']
    pb = np.mean([pin(y, o[f'{pre}{q}'], q).mean() for q in QUANTILES])
    iscore = ((u - l) + 10 * (l - y) * (y < l) + 10 * (y - u) * (y > u)).mean()
    return dict(n=len(o), cov80=((y >= l) & (y <= u)).mean(), cov95=((y >= l95) & (y <= u95)).mean(), avg_pinball=pb, interval_score80=iscore, width80=(u - l).mean())
sh_rows = {}; EV = {}; SHR = {}
for h in HORIZONS:
    best = None
    for sh in SHRINK_GRID:
        o = eval_intervals(h, BEST[h], sh); cvo = o[(o.index >= CV_START) & (o.end < HOLDOUT_START)]
        mt = metrics(cvo, 'q'); sh_rows[(h, sh)] = mt
        if best is None or mt['avg_pinball'] < best[0]: best = (mt['avg_pinball'], sh)
    SHR[h] = best[1]; EV[h] = eval_intervals(h, BEST[h], SHR[h])
display(pd.DataFrame(sh_rows).T.round(4)); print('Selected SHRINK:', SHR)

## 5. ประเมิน CV และ Holdout (vol-scaled conformal vs. constant-width conformal)

In [ ]:
rows = []
for h in HORIZONS:
    o = EV[h]; cvo = o[(o.index >= CV_START) & (o.end < HOLDOUT_START)]; ho = o[o.index >= HOLDOUT_START]
    for name, sub in [('CV', cvo), ('Holdout', ho), ('CV non-overlap', cvo.iloc[::h]), ('Holdout non-overlap', ho.iloc[::h])]:
        for pre, lab in [('q', 'vol-scaled'), ('c', 'constant')]:
            mt = metrics(sub, pre); mt.update(horizon=h, sample=name, method=lab, model=BEST[h]); rows.append(mt)
res = pd.DataFrame(rows)[['horizon','model','sample','method','n','cov80','cov95','avg_pinball','interval_score80','width80']]
res.to_csv(OUT / 'interval_metrics.csv', index=False); display(res.round(4))

# coverage by calendar year and by VIX regime (vol-scaled, 80% interval)
cy, cr = [], []
for h in HORIZONS:
    o = EV[h].dropna(subset=['ret', 'q0.1']).copy(); o['hit'] = (o.ret >= o['q0.1']) & (o.ret <= o['q0.9'])
    o = o[o.index >= CV_START]
    cy.append(o.groupby(o.index.year).hit.mean().rename(f'{h}d')); o['regime'] = pd.cut(o.vix, [0, 18, 25, 100], labels=['VIX<18', '18-25', '>25'])
    cr.append(o.groupby('regime').hit.mean().rename(f'{h}d'))
cov_year, cov_reg = pd.concat(cy, axis=1), pd.concat(cr, axis=1)
cov_year.to_csv(OUT / 'coverage80_by_year.csv'); cov_reg.to_csv(OUT / 'coverage80_by_vix.csv')
display(cov_year.round(3)); display(cov_reg.round(3)); print('เป้าหมาย coverage 80% — ดูว่าเบี่ยงเบนเป็นระบบในปี/ระดับ VIX ไหน')

## 6. Forecast ปัจจุบัน (refit ด้วยข้อมูลทั้งหมด) + forecast log

In [ ]:
fc_rows = []; P0 = close.iloc[-1]
for h in HORIZONS:
    tr_all = (END[h].notna() & YV[h].notna() & FH[h][HARE].notna().all(axis=1)).values
    base = {k: v.iloc[-1] for k, v in vol_predictions(h, LAST, LAST + pd.Timedelta(days=1), tr_all).items()}
    dd = D[h]; trn = dd[dd.y.notna()].dropna(subset=BASE); w, _ = nnls(trn[BASE].values, trn.y.values)
    base['stack'] = float(np.array([base[k] for k in BASE]) @ w)
    sig = np.exp(base[BEST[h]]) * np.sqrt(h); mu = SHR[h] * RET[h][tr_all].mean()
    zq = EV[h].iloc[-1][[f'z{q}' for q in QUANTILES]].values.astype(float)    # conformal z from realized windows only
    row = dict(as_of=LAST.date(), horizon=h, target_date=(LAST + pd.tseries.offsets.BDay(h)).date(), vol_model=BEST[h], price=P0, shrink=SHR[h],
               earnings_in_window=int(FH[h]['earn_in'].iloc[-1]), exp_daily_vol=np.exp(base[BEST[h]]), exp_annual_vol=np.exp(base[BEST[h]]) * np.sqrt(252), center_return=mu)
    for q, zz in zip(QUANTILES, zq): row[f'p{q*100:g}'] = P0 * np.exp(mu + zz * sig)
    fc_rows.append(row)
fc = pd.DataFrame(fc_rows); fc.to_csv(OUT / 'forecast_current.csv', index=False); display(fc.round(3))
# forecast log: save every run's forecast; later, upload the log and score it against realized prices (genuine out-of-sample test)
fc.to_csv(OUT / 'forecast_log.csv', index=False)
def score_log(path):
    lg = pd.read_csv(path, parse_dates=['as_of', 'target_date']); lg = lg[lg.target_date <= LAST]
    if lg.empty: print('ยังไม่มี forecast ที่ถึงวันครบกำหนด'); return None
    rows = []
    for t in lg.itertuples():
        act = close[close.index <= t.target_date].iloc[-1]; rows.append(dict(as_of=t.as_of.date(), horizon=t.horizon, actual=act, p10=t.p10, p90=t.p90, in_80=bool(t.p10 <= act <= t.p90), in_95=bool(t.p2_5 <= act <= t.p97_5) if hasattr(t, 'p2_5') else None))
    out = pd.DataFrame(rows); print(out); print('coverage80:', out.in_80.mean()); return out

## 7. กราฟ

In [ ]:
fig, axes = plt.subplots(len(HORIZONS), 1, figsize=(11, 3.6 * len(HORIZONS)))
for ax, h in zip(axes, HORIZONS):
    s = EV[h][EV[h].index >= HOLDOUT_START].dropna(subset=['ret', 'q0.1'])
    ax.fill_between(s.index, s['q0.025'], s['q0.975'], alpha=.15, label='95%'); ax.fill_between(s.index, s['q0.1'], s['q0.9'], alpha=.3, label='80%')
    ax.plot(s.index, s.ret, lw=1, color='k', label='actual log-return'); ax.set_title(f'Holdout {h}d: {BEST[h]} + rolling conformal'); ax.legend(loc='upper left', ncol=3)
plt.tight_layout(); plt.savefig(OUT / 'holdout_intervals.png', dpi=120); plt.show()
ax = cov_year.plot(kind='bar', figsize=(10, 3.5), title='80% interval coverage by year (target 0.80)'); ax.axhline(.8, color='k', ls='--'); plt.tight_layout(); plt.savefig(OUT / 'coverage_by_year.png', dpi=120); plt.show()
hist = close[-250:]; fig, ax = plt.subplots(figsize=(11, 4)); ax.plot(hist.index, hist.values, color='k', label='AMZN close')
for h, c_ in zip(HORIZONS, ['tab:blue', 'tab:orange', 'tab:green']):
    f = fc[fc.horizon == h].iloc[0]; x = LAST + pd.tseries.offsets.BDay(h)
    ax.vlines(x, f['p10'], f['p90'], color=c_, lw=6, alpha=.5, label=f'{h}d 80%'); ax.plot([x], [f['p50']], 'o', color=c_)
ax.legend(); ax.set_title('Current forecast ranges'); plt.tight_layout(); plt.savefig(OUT / 'forecast_ranges.png', dpi=120); plt.show()
archive = shutil.make_archive('AMZN_V4_Results', 'zip', OUT)
try:
    from google.colab import files; files.download(archive)
except ImportError: print('saved', archive)

## Limitations
- Holdout ของ V3/V4 ถูกดูผลแล้ว จึงไม่ใช่ holdout ที่แท้จริง — หลักฐานที่ซื่อสัตย์ที่สุดคือ `forecast_log.csv` ที่ตรวจกับราคาจริงภายหลัง (`score_log`)
- วันประกาศงบในอดีตใช้เป็น "ปฏิทินที่รู้ล่วงหน้า" (บริษัทประกาศล่วงหน้าราว 2 สัปดาห์) และวันงบถัดไปมาจาก Yahoo ซึ่งอาจเป็นวันประมาณการ
- เป้าหมายหลาย horizon ซ้อนทับกัน (ดูผล non-overlap และ DM ประกอบ)
- ใช้เฉพาะราคา/ตลาด/ปฏิทินงบ; ไม่มี implied vol หรือ intraday ซึ่งมักช่วยได้มากที่สุด
- ไม่ใช่คำแนะนำการลงทุน